[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/10_graph/10_proyecto_lightgcn_vs_mf.ipynb)

# Proyecto 10 · Candidatos por grafo para CineMatch: LightGCN (PyG) vs MF

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | 3 h |
| **GPU** | T4/L4 |
| **Unidades Colab** | ≈ 1–3 |
| **Prerrequisitos** | Lección 10 (y 05 para BPR) |

## 🏢 Contexto de negocio
El generador de candidatos de CineMatch es hoy un MF-BPR (módulo 05). El equipo de *growth* sospecha que los **usuarios nuevos o poco activos** reciben candidatos pobres.
Te piden evaluar si un **LightGCN** (propagación en grafo) mejora el *recall* de candidatos, sobre todo en la cola de usuarios poco activos, **sin** aumentar el coste de serving
(los embeddings se precalculan y se sirven con el mismo índice FAISS del módulo 08).

## 📦 Dataset
MovieLens-1M como feedback implícito. **Split temporal por usuario**: el último 20 % de las interacciones de cada usuario es test; del train, el último 10 % se usa como validación para elegir épocas/capas.

## ✅ Entregables y rúbrica
| Criterio | Objetivo |
|---|---|
| MF-BPR y LightGCN (PyG) entrenados con el **mismo** bucle, pérdida y presupuesto | obligatorio |
| Selección de K ∈ {1,2,3,4} y épocas por validación (no por test) | obligatorio |
| Mejora relativa de Recall@20 de LightGCN sobre MF en test | ≥ +3 % (orientativo; con MF bien regularizado puede ser menor: repórtalo honestamente) |
| Análisis por cuartil de actividad | gráfico + conclusión |
| Exportación de embeddings a un índice FAISS y latencia de top-100 | p50 < 5 ms por consulta en CPU |

In [ ]:
# PyTorch Geometric (sin extensiones compiladas: LightGCN sólo necesita el paquete base) + gensim + networkx
!pip install -q torch_geometric gensim networkx

!pip install -q faiss-cpu

In [ ]:
import math, time, random, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = False          # True = pocas épocas para iterar; False = experimento completo en GPU
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("device:", device, "| torch", torch.__version__)

## 🧰 Utilidades mínimas (vienen de los módulos 01 y 02)

Para que este notebook sea **autocontenido en Colab**, copiamos aquí una versión reducida de las utilidades que construimos en
`01_data` (carga de MovieLens, secuencias, *splits*) y `02_evaluation` (Recall@K y NDCG@K con *full ranking* y split temporal por usuario).

- Si la descarga falla (sin red, *firewall*…), se generan **datos sintéticos con estructura secuencial** para que el notebook nunca se bloquee. Los números con datos sintéticos **no** son comparables con los papers.
- Convención de IDs: los ítems se re-indexan a `1..n_items` y **el 0 es el *padding***.

In [ ]:
# ==== Utilidades (versión mínima de 01_data) ====
import os, io, zipfile, urllib.request

ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

def make_synthetic_ml(n_users=1500, n_items=700, n_genres=12, seed=42):
    '''Datos de juguete con señal secuencial: "siguiente capítulo", deriva de género y popularidad.'''
    rng = np.random.default_rng(seed)
    genre = rng.integers(0, n_genres, n_items)
    by_genre = [np.where(genre == g)[0] for g in range(n_genres)]
    pop = np.minimum(rng.zipf(1.4, n_items), 60).astype(float); pop /= pop.sum()
    rows = []
    for u in range(1, n_users + 1):
        g = int(rng.integers(n_genres)); it = int(rng.choice(by_genre[g])); t = 978_300_000 + int(rng.integers(0, 10**7))
        for _ in range(int(rng.integers(20, 160))):
            rows.append((u, it + 1, int(rng.integers(3, 6)), t))
            t += int(rng.integers(30, 3 * 86400)); r = rng.random()
            if r < 0.45:   it = (it + int(rng.integers(1, 4))) % n_items          # secuela / siguiente capítulo
            elif r < 0.8:  it = int(rng.choice(by_genre[g]))                       # mismo género
            else:          g = int(rng.integers(n_genres)); it = int(rng.choice(n_items, p=pop))
    ratings = pd.DataFrame(rows, columns=["user", "item", "rating", "ts"]).drop_duplicates(["user", "item"])
    names = ["Action", "Comedy", "Drama", "Horror", "Sci-Fi", "Romance", "Thriller",
             "Animation", "Documentary", "Crime", "Musical", "War"]
    movies = pd.DataFrame({"item": np.arange(1, n_items + 1),
                           "title": [f"Película sintética {i} (2000)" for i in range(1, n_items + 1)],
                           "genres": [names[g % len(names)] for g in genre]})
    return ratings, movies

def load_movielens_1m(data_dir="data"):
    '''Devuelve (ratings[user,item,rating,ts], movies[item,title,genres]).'''
    os.makedirs(data_dir, exist_ok=True)
    zpath = os.path.join(data_dir, "ml-1m.zip")
    try:
        if not os.path.exists(zpath):
            urllib.request.urlretrieve(ML1M_URL, zpath)
        with zipfile.ZipFile(zpath) as z:
            ratings = pd.read_csv(z.open("ml-1m/ratings.dat"), sep="::", engine="python",
                                  names=["user", "item", "rating", "ts"])
            movies = pd.read_csv(z.open("ml-1m/movies.dat"), sep="::", engine="python",
                                 names=["item", "title", "genres"], encoding="latin-1")
        return ratings, movies
    except Exception as e:
        if os.path.exists(zpath):
            os.remove(zpath)
        print(f"⚠️ No se pudo descargar MovieLens-1M ({type(e).__name__}). Uso datos SINTÉTICOS.")
        return make_synthetic_ml()

In [ ]:
# ==== Split temporal por usuario y Recall/NDCG@K multi-relevante (01 y 02) ====
import scipy.sparse as sp

def temporal_user_split(df, test_frac=0.2):
    '''Para cada usuario, el último `test_frac` de sus interacciones (por tiempo) va a test.'''
    df = df.sort_values(["u", "ts"], kind="mergesort")
    rank = df.groupby("u").cumcount()
    n = df.groupby("u")["i"].transform("size")
    is_test = rank >= np.ceil(n * (1 - test_frac))
    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)

def to_csr(df, n_users, n_items):
    return sp.csr_matrix((np.ones(len(df), dtype=np.float32), (df.u.values, df.i.values)), shape=(n_users, n_items))

@torch.no_grad()
def evaluate_topk(score_fn, train_csr, test_csr, k=20, batch_size=1024):
    '''score_fn(user_ids LongTensor) -> Tensor [B, n_items]. Enmascara train. Devuelve Recall@K y NDCG@K medios.'''
    users = np.where(np.asarray((test_csr > 0).sum(1)).ravel() > 0)[0]
    rec, ndcg = [], []
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    for b in range(0, len(users), batch_size):
        u = users[b:b + batch_size]
        s = score_fn(torch.as_tensor(u)).float().cpu().numpy()
        s[train_csr[u].nonzero()] = -np.inf                     # no recomendar lo visto en train
        top = np.argpartition(-s, k, axis=1)[:, :k]
        top = np.take_along_axis(top, np.argsort(-np.take_along_axis(s, top, 1), 1), 1)
        te = test_csr[u].toarray() > 0
        hits = np.take_along_axis(te, top, 1)
        n_rel = te.sum(1)
        rec.append(hits.sum(1) / n_rel)
        idcg = np.array([disc[:min(int(r), k)].sum() for r in n_rel])
        ndcg.append((hits * disc).sum(1) / idcg)
    return {f"Recall@{k}": float(np.concatenate(rec).mean()), f"NDCG@{k}": float(np.concatenate(ndcg).mean())}

In [ ]:
ratings, movies = load_movielens_1m()
# Feedback implícito: toda valoración es una interacción positiva (protocolo habitual de LightGCN/NGCF)
users_u = np.sort(ratings.user.unique()); items_u = np.sort(ratings.item.unique())
u2i = {u: k for k, u in enumerate(users_u)}; it2i = {it: k for k, it in enumerate(items_u)}
df = pd.DataFrame({"u": ratings.user.map(u2i), "i": ratings.item.map(it2i), "ts": ratings.ts})
n_users, n_items = len(users_u), len(items_u)
train_df, test_df = temporal_user_split(df, test_frac=0.2)
test_df = test_df[test_df.i.isin(set(train_df.i))]                 # ítems nunca vistos en train no son predecibles
R_train, R_test = to_csr(train_df, n_users, n_items), to_csr(test_df, n_users, n_items)
title = dict(zip(movies.item.map(it2i), movies.title)); genre = dict(zip(movies.item.map(it2i), movies.genres.str.split("|").str[0]))
print(f"usuarios={n_users:,} ítems={n_items:,} train={R_train.nnz:,} test={R_test.nnz:,} densidad={R_train.nnz/(n_users*n_items):.3%}")

In [ ]:
# Validación: último 10 % del train de cada usuario
fit_df, val_df = temporal_user_split(train_df, test_frac=0.1)
val_df = val_df[val_df.i.isin(set(fit_df.i))]
R_fit, R_val = to_csr(fit_df, n_users, n_items), to_csr(val_df, n_users, n_items)
print(f"fit={R_fit.nnz:,}  val={R_val.nnz:,}  test={R_test.nnz:,}")

## Paso 1 · Grafo para PyG
Construye `edge_index` **no dirigido** (usuarios `0..n_users-1`, ítems desplazados `+ n_users`) a partir de una matriz CSR de entrenamiento.

In [ ]:
def build_edge_index(R):
    # TODO: devuelve (edge_index [2, 2*nnz] no dirigido, train_pairs [2, nnz] usuario->ítem desplazado)
    raise NotImplementedError

## Paso 2 · Entrenamiento común (MF y LightGCN)
Truco: **MF = LightGCN de PyG con `num_layers=0`**. Escribe una única función `fit(K, R_train, R_eval, epochs)` que entrene con `recommendation_loss` y devuelva el modelo y su historial de Recall@20 en `R_eval`.

In [ ]:
def fit(K, R_train, R_eval, epochs=60, d=64, lr=2e-3, batch_size=8192, eval_every=10):
    # TODO
    raise NotImplementedError

## Paso 3 · Selección por validación, evaluación en test y análisis por actividad
1. Para K ∈ {0 (MF), 1, 2, 3, 4} entrena en `R_fit` y evalúa en `R_val`. 2. Elige K* y reentrena en `R_train` completo. 3. Evalúa en test MF vs LightGCN-K*. 4. Desglosa por cuartil de actividad.

In [ ]:
# TODO: bucle de selección de K y tabla de resultados
...

## Paso 4 · Serving con FAISS
Exporta los embeddings de ítems a un `faiss.IndexFlatIP` y mide la latencia de top-100 para 1.000 usuarios (consulta a consulta).

In [ ]:
# TODO: índice FAISS + medición de latencia p50/p99
...

---
# ⛔ SPOILER — Solución de referencia

In [ ]:
from torch_geometric.nn import LightGCN as PyGLightGCN

def build_edge_index(R):
    coo = R.tocoo()
    src = torch.from_numpy(coo.row).long(); dst = torch.from_numpy(coo.col).long() + n_users
    return torch.stack([torch.cat([src, dst]), torch.cat([dst, src])]).to(device), torch.stack([src, dst]).to(device)

def pyg_score_fn(model, edge_index):
    with torch.no_grad():
        emb = model.get_embedding(edge_index)
    U, I = emb[:n_users], emb[n_users:]
    return (lambda u: U[u.to(device)] @ I.T), U, I

def fit(K, R_train, R_eval, epochs=60, d=64, lr=2e-3, batch_size=8192, eval_every=10):
    torch.manual_seed(seed)
    ei, pairs = build_edge_index(R_train)
    model = PyGLightGCN(num_nodes=n_users + n_items, embedding_dim=d, num_layers=K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr); hist = collections.defaultdict(list)
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(pairs.size(1), device=device)
        for b in range(0, len(perm), batch_size):
            pos = pairs[:, perm[b:b + batch_size]]
            neg = torch.stack([pos[0], torch.randint(n_users, n_users + n_items, (pos.size(1),), device=device)])
            pr, nr = model(ei, torch.cat([pos, neg], 1)).chunk(2)
            loss = model.recommendation_loss(pr, nr, node_id=torch.cat([pos, neg], 1).unique())
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % eval_every == 0 or ep == epochs:
            model.eval(); r = evaluate_topk(pyg_score_fn(model, ei)[0], R_train, R_eval)
            hist["epoch"].append(ep); hist["Recall@20"].append(r["Recall@20"])
    return model, ei, dict(hist)

In [ ]:
EPOCHS = 3 if FAST_DEV_RUN else 60
EVAL_EVERY = 1 if FAST_DEV_RUN else 10
sel = {}
for K in [0, 1, 2, 3, 4]:
    _, _, h = fit(K, R_fit, R_val, epochs=EPOCHS, eval_every=EVAL_EVERY)
    best = int(np.argmax(h["Recall@20"]))
    sel[K] = {"best_epoch": h["epoch"][best], "val_Recall@20": h["Recall@20"][best]}
    print(K, sel[K])
sel_df = pd.DataFrame(sel).T; display(sel_df)
K_star = int(sel_df.loc[1:, "val_Recall@20"].idxmax())
print("K* =", K_star)

In [ ]:
final, fitted = {}, {}
for name, K in [("MF-BPR", 0), (f"LightGCN K={K_star}", K_star)]:
    ep = int(sel_df.loc[K, "best_epoch"])
    model, ei, _ = fit(K, R_train, R_test, epochs=ep, eval_every=ep)
    fn, U, I = pyg_score_fn(model, ei)
    fitted[name] = (fn, U, I)
    final[name] = evaluate_topk(fn, R_train, R_test)
fd = pd.DataFrame(final).T; display(fd.round(4))
mf, lg = fd.iloc[0]["Recall@20"], fd.iloc[1]["Recall@20"]
print(f"Mejora relativa Recall@20: {100*(lg-mf)/mf:+.2f} %")

In [ ]:
act = np.diff(R_train.indptr)
buckets = pd.qcut(act, 4, labels=["Q1 (poca)", "Q2", "Q3", "Q4 (mucha)"])
rows = []
for name, (fn, _, _) in fitted.items():
    for bk in buckets.categories:
        m = np.zeros(n_users, bool); m[np.where(buckets == bk)[0]] = True
        rows.append({"modelo": name, "cuartil": bk, **evaluate_topk(fn, R_train, R_test.multiply(m[:, None]).tocsr())})
bq = pd.DataFrame(rows)
piv = bq.pivot(index="cuartil", columns="modelo", values="Recall@20")
lg_name = [c for c in piv.columns if c.startswith("LightGCN")][0]
piv["mejora_%"] = 100 * (piv[lg_name] - piv["MF-BPR"]) / piv["MF-BPR"]
display(piv.round(4))
fig, ax = plt.subplots(figsize=(8, 3.6))
sns.barplot(data=bq, x="cuartil", y="Recall@20", hue="modelo", ax=ax); ax.set(title="Recall@20 por actividad"); plt.show()

In [ ]:
import faiss
name_lg = [k for k in fitted if k.startswith("LightGCN")][0]
_, U, I = fitted[name_lg]
I_np = I.detach().float().cpu().numpy().astype("float32"); U_np = U.detach().float().cpu().numpy().astype("float32")
index = faiss.IndexFlatIP(I_np.shape[1]); index.add(I_np)
lat = []
for u in np.random.default_rng(seed).choice(n_users, 1000):
    t0 = time.perf_counter(); index.search(U_np[u:u + 1], 100); lat.append((time.perf_counter() - t0) * 1e3)
print(f"FAISS top-100: p50 = {np.percentile(lat, 50):.3f} ms   p99 = {np.percentile(lat, 99):.3f} ms  (ítems={len(I_np):,})")

**Lectura esperada.** LightGCN suele mejorar más en los cuartiles de **poca actividad** (Q1–Q2), que toman prestada señal de vecinos a 2–3 saltos; en Q4 la diferencia es pequeña.
El coste de serving no cambia: siguen siendo dos tablas de embeddings y un producto escalar en FAISS. El coste extra está **offline** (propagación en el entrenamiento).

## 🚀 Retos extra
1. **Nodos de género** (grafo heterogéneo plano): añade los géneros como nodos extra conectados a sus películas y vuelve a medir Recall@20 en ítems poco populares.
2. **LightGCN post-hoc**: entrena MF y aplica la propagación $\frac{1}{K+1}\sum_k \hat A^k E$ **sólo en inferencia**. ¿Cuánta ganancia recupera sin coste de entrenamiento?
3. **GF-CF** (Shen et al. 2021): implementa el filtro lineal sin entrenamiento sobre $\tilde R^\top \tilde R$ y compáralo.
4. **Negativos duros**: muestrea la mitad de los negativos del top-100 del propio modelo (estilo *curriculum* de PinSage).
5. **Escala**: MovieLens-32M en A100 con `torch_geometric` + `NeighborLoader` en lugar de propagación completa.

## 🤔 Reflexión (producción / MLOps)
- Los embeddings de LightGCN dependen de **todo el grafo**: si reentrenas cada noche, ¿cómo evitas que el índice FAISS «salte» y cambien los candidatos de todos los usuarios (*embedding drift*, módulo 17)?
- ¿Qué harías con un usuario que se registró hace 10 minutos? (pista: DeepWalk/item2vec sobre su sesión, módulo 09, o features inductivas tipo PinSage).